# splitql — quickstart

`splitql` parte una query SQL en N fragmentos por partición (con agregación parcial two-phase) más una query `reduce`. Planificación pura: la ejecución la pones tú.

Este notebook: dataset de demo → plan básico → mini-runner + comparación con el oráculo single-node → pruning por stats → recomendación de workers → queries no elegibles → grafo interactivo → envelope JSON.

Lanzar con: `.venv/bin/jupyter lab examples/quickstart.ipynb` desde la raíz del repo.

In [ ]:
import duckdb
import pyarrow as pa

import splitql
from splitql import plan, DataFile, ColumnStats, DuckLakeSource

print("splitql", splitql.__version__, "| duckdb", duckdb.__version__)

## 1. Dataset de demo

Seis ficheros Parquet mensuales (enero-junio 2026), como los que dejaría una ingesta particionada por fecha. Los rangos de fecha disjuntos son ideales para ver el pruning.

In [ ]:
from pathlib import Path

DATA = Path("_demo_data")
DATA.mkdir(exist_ok=True)

con = duckdb.connect()
files = []
for month in range(1, 7):
    path = str(DATA / f"sales-2026-{month:02d}.parquet")
    con.execute(f"""
        COPY (
            SELECT (i + {month} * 100000) AS order_id,
                   ['north', 'south', 'east', 'west'][1 + i % 4] AS region,
                   round(10 + (i * 37 % 990) / 7.0, 2) AS amount,
                   DATE '2026-{month:02d}-01' + INTERVAL (i % 28) DAY AS d
            FROM range(0, 50000 + {month} * 10000) r(i)
        ) TO '{path}' (FORMAT parquet)
    """)
    files.append(path)

files

## 2. Plan básico

SQL + lista de ficheros → fragmentos + reduce. Sin conexiones, sin ejecución: solo strings de SQL.

In [ ]:
sql = """
SELECT region, count(*) AS orders, sum(amount) AS revenue, avg(amount) AS avg_ticket
FROM sales
WHERE amount > 20
GROUP BY region
ORDER BY revenue DESC
"""

p = plan(sql, files=files, workers=3)

print("eligible:", p.eligible, "| workers:", p.workers)
print("\n--- fragment 0 ---\n", p.fragments[0][:300], "...")
print("\n--- reduce ---\n", p.reduce)

Fíjate en la descomposición two-phase: `avg(amount)` viaja como el par `SUM`/`COUNT` (`a2_s`, `a2_c`) y el reduce lo recombina como división. Las claves de grupo van como `g0..gk` para no colisionar nunca con columnas del usuario.

## 3. Ejecutar el plan (mini-runner) y comparar con el oráculo

El runner mínimo: cada fragmento en su propia conexión (en producción serían workers remotos), concatenar como Arrow, registrar como `partials`, ejecutar el reduce. Y el contrato: mismo resultado que single-node.

In [ ]:
def run_plan(p):
    if not p.eligible:
        raise ValueError(f"not splittable: {p.reason} — run p.query single-node")
    partials = pa.concat_tables(
        [duckdb.connect().execute(f).to_arrow_table() for f in p.fragments]
    )
    c = duckdb.connect()
    c.register(p.partials_table, partials)
    return c.execute(p.reduce).fetchdf()

distributed = run_plan(p)
distributed

In [ ]:
# el oráculo: la misma query, single-node, sobre todos los ficheros
c = duckdb.connect()
c.execute(f"CREATE VIEW sales AS SELECT * FROM read_parquet({files})")
single_node = c.execute(sql).fetchdf()

import pandas as pd

# check_dtype=False: partial COUNTs re-aggregate as SUM -> HUGEINT, single-node COUNT is BIGINT
pd.testing.assert_frame_equal(distributed, single_node, check_dtype=False, check_exact=False)
print("✓ resultado idéntico a single-node")
single_node

## 4. Zone-map pruning

Si los `DataFile` llevan stats min/max por columna, el planner descarta ficheros que el WHERE nunca puede tocar, ANTES de agrupar. Aquí sacamos las stats reales con DuckDB (una query por fichero; en DuckLake ya vienen en el catálogo).

In [ ]:
import os

stat_files = []
for f in files:
    lo, hi, n = duckdb.connect().execute(
        f"SELECT min(d), max(d), count(*) FROM read_parquet('{f}')"
    ).fetchone()
    stat_files.append(DataFile(
        f,
        size_bytes=os.path.getsize(f),
        stats={"d": ColumnStats(lo, hi)},
        row_count=n,
    ))

p2 = plan(
    "SELECT count(*) AS n, sum(amount) AS s FROM sales WHERE d >= DATE '2026-05-01'",
    files=stat_files,
)
print("fragments:", p2.workers)
print("pruned:", p2.pruned_files)
run_plan(p2)

Cuatro de los seis ficheros (enero-abril) quedan podados: menos fragmentos, menos workers, menos I/O. La poda es conservadora: sin stats, o ante un predicado que no sabe demostrar, el fichero se queda (mantener nunca es incorrecto).

## 5. Recomendación de workers

Con tamaños conocidos, si no pasas `workers` el planner recomienda: `ceil(bytes_totales / objetivo)`, donde el objetivo sale de `worker_memory_bytes / 4` (o 512MB por defecto), con tope en el número de ficheros y en `max_workers`. El agrupado balancea por tamaño (LPT).

In [ ]:
for kwargs in [dict(), dict(worker_memory_bytes=2 * 2**30), dict(max_workers=2)]:
    pn = plan("SELECT count(*) FROM sales", files=stat_files, **kwargs)
    sizes = [sum(f.size_bytes for f in g) // 1024 for g in pn.fragment_files]
    print(f"{str(kwargs):45s} -> {pn.workers} workers, KB por fragmento: {sizes}")

## 6. Lo que NO es elegible

Whitelist conservadora: lo que no se puede partir con garantía de corrección devuelve `eligible=False` con la razón (nunca una excepción, nunca un split incorrecto). El fallback es ejecutar `p.query` single-node.

In [ ]:
for q in [
    "SELECT a.*, b.x FROM a JOIN b ON a.id = b.id",
    "SELECT count(DISTINCT region) FROM sales",
    "SELECT region, count(*) FROM sales GROUP BY region HAVING count(*) > 10",
    "SELECT id, row_number() OVER (ORDER BY d) FROM sales",
    "SELECT count(*) FROM sales WHERE random() < 0.5",
]:
    r = plan(q, files=files, workers=2)
    print(f"{'✗' if not r.eligible else '?'} {r.reason:60s} | {q[:48]}")

## 7. Grafo interactivo del plan

`p.to_html()` genera una página autocontenida (sin assets externos): cards por fragmento con ficheros, share de bytes y SQL expandible, flujo scan → partials → reduce → result. También hay `p.to_dot()` para Graphviz.

In [ ]:
from IPython.display import IFrame, HTML

html_page = p2.to_html()
Path("plan.html").write_text(html_page)
print("guardado en examples/plan.html — ábrelo en el navegador, o inline:")
HTML(f'<iframe srcdoc="{html_page.replace(chr(34), "&quot;")}" style="width:100%;height:520px;border:1px solid #ddd;border-radius:8px"></iframe>')

## 8. El envelope JSON

Todo el plan viaja como JSON: un coordinador no-Python (Go, Rust, un RayJob...) puede consumirlo tal cual. Los planes no elegibles conservan `query` para el fallback.

In [ ]:
import json
print(json.dumps(p2.to_dict(), indent=2, default=str)[:1200], "...")

## 9. DuckLake (receta)

Con una lake attachada, la fuente sale del catálogo y hereda sus guardas de corrección: delete files → no elegible; datos inlined sin verificar → no elegible (fail-closed, exige `has_inlined_data=False` explícito).

```python
rows = con.execute("FROM ducklake_list_files('lake', 'sales')").fetchall()
src = DuckLakeSource.from_list_files(rows, has_inlined_data=False)
p = plan(sql, source=src)
```

Las stats para pruning están en la tabla `ducklake_file_column_stats` del catálogo.